In [ ]:
import concurrent.futures
import json
import logging
import os
import sys
import threading

from pathlib import Path

basepath = Path(
    r"C:\Users\s158699\Documents\PhD\Experiments\Simulation\aggregated_event_data"
)

sys.path.append(os.path.join(*basepath.parts))

from aggregated_event_data import simulate
from numpy import isclose
from pathlib import Path

logging.basicConfig(level=logging.WARNING, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)
from aggregated_traces.utils.ekg_analysis import (
    compute_aggregation_uniformity,
    compute_aggregation_average_kl_trace_graph,
    compute_trace_probabilities,
    compute_number_of_merges_in_trace_graph,
    compute_number_of_steps_at_aggregations,
)

In [ ]:
from rdflib import Graph, URIRef, Variable
from typing import List


def select_packing_units_with_quality_issue(
    graph: Graph, threshold: float
) -> List[URIRef]:
    """
    Find packing unit(s) with average quality below a certain threshold.
    """

    query_lowest_quality = """
    PREFIX : <http://example.org/def/ekg/aggregated_traces/>
    SELECT DISTINCT
        ?packing_unit
        (sum(?device_quality)/count(?device_quality) as ?average_quality)
    WHERE {
        [] :bizStep "packing" ;
            :parentEntity ?packing_unit ;
            :device/:quality ?device_quality .

        ?packing_unit a :PackingUnit .
    }
    GROUP BY ?packing_unit
    """

    bindings = graph.query(query_lowest_quality).bindings

    return [
        b[Variable("packing_unit")]
        for b in bindings
        if b[Variable("average_quality")].toPython() < threshold
    ]


from pandas import DataFrame, Index, isna


def get_entity_record_number(df: DataFrame, entity: str) -> int:
    try:
        return Index(df["entity_target"]).get_loc(entity) + 1
    except KeyError:
        return None


def compute_trace_results(
    event_data_file: str,
    quality_threshold: float,
    result_files: Path,
    run_file_name: str,
) -> str | None:
    # Parse data to RDF graph
    ekg_graph_rdf = load_rdf_graph(event_data_file, store="Oxigraph")

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    # query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)
    print("ekg_graph_nx")

    # Find packing units with lowest (average) quality for which to find the root cause (entity)
    entities_source_backward = select_packing_units_with_quality_issue(
        graph=ekg_graph_rdf, threshold=quality_threshold
    )
    print("select_packing_units_with_quality_issue")

    # Compute backward trace probabilities
    try:
        df_backward, edges_paths_backward = compute_trace_probabilities(
            rdf_trace_graph=ekg_graph_rdf,
            nx_trace_graph=ekg_graph_nx,
            source_entities=entities_source_backward,
            trace_backward=True,
        )
    except RuntimeError as e:
        logger.warning(e)
        return None
    print("df_backward", len(df_backward))

    # Compute statistics on trace graphs for source-target node pairs
    aggregation_kl = compute_aggregation_uniformity(graph=ekg_graph_rdf)
    for i in df_backward.index:
        # Compute number of merges in the trace graph
        df_backward.loc[i, "n_merges"] = compute_number_of_merges_in_trace_graph(
            trace_graph=ekg_graph_nx,
            source=df_backward.loc[i, "node_source"],
            target=df_backward.loc[i, "node_target"],
        )

        # Compute uniformity of splits and merges
        df_backward.loc[i, "split_merge_kl"] = (
            compute_aggregation_average_kl_trace_graph(
                trace_graph=ekg_graph_nx,
                aggregation_kl=aggregation_kl,
                source=df_backward.loc[i, "node_source"],
                target=df_backward.loc[i, "node_target"],
            )
        )
        print("df_backward", i)

    # Get the number of steps executed at the aggregation events
    steps_aggregations = compute_number_of_steps_at_aggregations(
        graph=ekg_graph_rdf, events=df_backward["node_source"].unique()
    )
    df_backward["n_production_steps_aggregations"] = df_backward["node_source"].map(
        steps_aggregations
    )
    print("n_production_steps_aggregations", i)

    # Validate if probabilities add up to one
    if not all(
        isclose(
            df_backward.groupby(["entity_source", "product_model"])["probability"]
            .sum()
            .astype(float),
            1,
        )
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    df_backward["probability"] = df_backward["probability"].astype(float)
    with lock:
        with open(result_files.joinpath(run_file_name), "w") as f:
            json.dump(
                df_backward.to_dict(orient="records"),
                f,
            )

    ekg_graph_rdf.close()

    return result_files.joinpath(run_file_name)

In [ ]:
from collections import defaultdict
from typing import Dict, Tuple


def generate_scenario(
    output_file: str,
    n_lots: int,
    n_devices: List[int],
    steps_resources: Dict[str, Tuple[int]],
    root_cause_resource: str,
    required_material: Dict[str, str] = None,
    merge_after_steps: List[str] = [],
    # n_lots_merge: int = 2,
    split_configs: List[dict] = [],
) -> str:
    # Select lots for merge after a certain step
    merge_configs = [{"after_step": step} for step in merge_after_steps]
    merge_configs_lots = defaultdict(list)
    for config in merge_configs:
        # for i in [i for i in range(0, n_lots, n_lots_merge)]:
        for i in range(n_lots):
            merge_configs_lots[f"Lot{i}"].append(config)

    production_lots = []
    for i in range(n_lots):
        lot_id = f"Lot{i}"
        production_lot = {
            "id": lot_id,
            "steps": list(steps_resources.keys()),
            "merge": merge_configs_lots.get(lot_id, []),
            "split": split_configs,
            "n_devices": n_devices[i],
        }

        if required_material:
            production_lot["required_material"] = required_material

        production_lots.append(production_lot)

    production_resources = []
    for step, (n, mean_duration) in steps_resources.items():
        production_resources.extend(
            [
                {
                    "id": f"{step}{i}",
                    "step": step,
                    "mean_move": 0.5,
                    "mean_duration": mean_duration,
                    "mean_breakdown": 5,
                    "mean_repair": 10,
                    "process_yield": 0.5 if root_cause_resource == f"{step}{i}" else 1,
                }
                for i in range(n)
            ]
        )

    simulation_config = {
        "production_lots": production_lots,
        "production_resources": production_resources,
        "material_lot_size": 100,
        "packing_unit_size": 50,
    }

    # with lock:
    with open(output_file, "w") as f:
        json.dump(simulation_config, f, indent=2)

    return json.dumps(simulation_config)

# Generate (simulate) and process event logs

In [ ]:
import pyDOE2

from itertools import chain, combinations


def all_subsets(iterable):
    "Returns all possible subsets of a given iterable"
    s = list(iterable)
    return list(chain.from_iterable(combinations(s, r) for r in range(len(s) + 1)))


def generate_non_overlapping_pairs(subsets):
    pairs = []
    for subset1, subset2 in combinations(subsets, 2):
        if not set(subset1) & set(subset2):  # Ensure no overlapping values
            pairs.append((subset1, subset2))
    return pairs


steps = ["WT", "DB", "WB", "Marking", "FT"]
subsets_of_steps = all_subsets(steps)
split_merge_after_steps_values = generate_non_overlapping_pairs(subsets_of_steps)

number_of_devices_values = [
    [50] * 32,
    [25, 75] * 16,
    [10, 90] * 16,
    [25, 25, 25, 125] * 8,
    [10, 10, 10, 170] * 8,
]

n_resources_factor_values = [
    2,
    4,
    8,
    16,
]

# Generate Design of Experiments (DoE) with reduction
levels = [
    len(split_merge_after_steps_values),
    len(number_of_devices_values),
    len(n_resources_factor_values),
]
reduction = 5

doe = pyDOE2.gsd(levels, reduction)

## Compute trace probabilities

### Local

In [ ]:
n_runs = 1

# Define the quality threshold
quality_threshold = 0.9

merge_after_steps = ["FT"]
split_after_steps = ["WT", "DB", "WB", "Marking"]
number_of_devices = [10, 90] * 16
n_resources_factor = 8

number_of_devices_label = set(str(n) for n in number_of_devices)
scenario_name = (
    "backward/scenarios/doe/"
    + f"merge_after_steps={'-'.join(merge_after_steps)}"
    + f"+split_after_steps={'-'.join(split_after_steps)}"
    + f"+number_of_devices={'-'.join(number_of_devices_label)}"
    + f"+n_resources_factor={n_resources_factor}"
)

split_configs = [
    {"after_step": step, "number_of_split_lots": 2} for step in split_after_steps
]

generate_scenario(
    output_file=scenario_name + ".json",
    n_lots=32,
    n_devices=number_of_devices,
    steps_resources={
        "WT": (n_resources_factor * 2, 1),
        "DB": (n_resources_factor * 2, 2),
        "WB": (n_resources_factor * 4, 20),
        "Marking": (n_resources_factor * 4, 1),
        "FT": (n_resources_factor * 2, 1),
    },
    root_cause_resource="WB1",
    # required_material: Dict[str, str] = None,
    merge_after_steps=list(merge_after_steps),
    # n_lots_merge: int = 2,
    split_configs=split_configs,
)

event_log_files = basepath.joinpath(f"logs/{scenario_name}/")
if not event_log_files.exists():
    os.mkdir(event_log_files)

result_files = Path(f"output/{scenario_name}/")
if not result_files.exists():
    os.mkdir(result_files)

lock = threading.Lock()

for i in range(n_runs):
    # if i != 0:
    #     continue

    run_file_name = f"run_{i}.json"
    event_data_file = event_log_files.joinpath(run_file_name)
    simulate.main(
        config_file=scenario_name + ".json",
        runtime=10000,
        output_event_log_file=event_data_file,
    )

    print(event_data_file)

    compute_trace_results(
        event_data_file, quality_threshold, result_files, run_file_name
    )

### Local (multiple threads)

In [ ]:
def worker(scenario_name, event_log_files, i, quality_threshold, result_files):
    run_file_name = f"run_{i}.json"
    event_data_file = event_log_files.joinpath(run_file_name)
    simulate.main(
        config_file=scenario_name + ".json",
        runtime=10000,
        output_event_log_file=event_data_file,
    )

    print(event_data_file)

    compute_trace_results(
        event_data_file, quality_threshold, result_files, run_file_name
    )

    return scenario_name


n_runs = 10

# Define the quality threshold
quality_threshold = 0.9

# Experiment 1: merge after different production steps
# variable = "merge_after_steps"
# variable_values = [
#     ["WT"],
#     ["WT", "DB"],
#     ["WT", "DB", "WB"],
#     ["WT", "DB", "WB", "Marking"],
#     ["WT", "DB", "WB", "Marking", "FT"],
# ]

# Experiment 2: number of devices in (initiated) production lots
# variable = "number_of_devices"
# variable_values = [
#     [50] * 32,
#     [25, 75] * 16,
#     [10, 90] * 16,
#     [25, 25, 25, 125] * 8,
#     [10, 10, 10, 170] * 8,
# ]

# Experiment 3: number of resources
# variable = "n_resources_factor"
# variable_values = [
#     2,
#     4,
#     8,
#     16,
# ]

# Experiment 4: split after different production steps
# variable = "split_after_steps"
# variable_values = [
#     # ["WT"],
#     # ["WT", "DB"],
#     # ["WT", "DB", "WB"],
#     # ["WT", "DB", "WB", "Marking"],
#     ["WT", "DB", "WB", "Marking", "FT"],
# ]

# merge_after_steps = []
# number_of_devices = [50] * 32
# number_of_devices_label = set(str(n) for n in number_of_devices)
# n_resources_factor = 1

lock = threading.Lock()

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    for d_i in range(len(doe[:4])):
        merge_after_steps, split_after_steps = split_merge_after_steps_values[
            doe[d_i][0]
        ]
        number_of_devices = number_of_devices_values[doe[d_i][1]]
        n_resources_factor = n_resources_factor_values[doe[d_i][2]]

        number_of_devices_label = set(str(n) for n in number_of_devices)
        scenario_name = (
            "backward/scenarios/doe/"
            + f"merge_after_steps={'-'.join(merge_after_steps)}"
            + f"+split_after_steps={'-'.join(split_after_steps)}"
            + f"+number_of_devices={'-'.join(number_of_devices_label)}"
            + f"+n_resources_factor={n_resources_factor}"
        )

        split_configs = [
            {"after_step": step, "number_of_split_lots": 2}
            for step in split_after_steps
        ]

        generate_scenario(
            output_file=scenario_name + ".json",
            n_lots=32,
            n_devices=number_of_devices,
            steps_resources={
                "WT": (n_resources_factor * 2, 1),
                "DB": (n_resources_factor * 2, 2),
                "WB": (n_resources_factor * 4, 20),
                "Marking": (n_resources_factor * 4, 1),
                "FT": (n_resources_factor * 2, 1),
            },
            root_cause_resource="WB1",
            # required_material: Dict[str, str] = None,
            merge_after_steps=list(merge_after_steps),
            # n_lots_merge: int = 2,
            split_configs=split_configs,
        )

        event_log_files = basepath.joinpath(f"logs/{scenario_name}/")
        if not event_log_files.exists():
            os.mkdir(event_log_files)

        result_files = Path(f"output/{scenario_name}/")
        if not result_files.exists():
            os.mkdir(result_files)

        future_to_url = {
            executor.submit(
                worker,
                scenario_name,
                event_log_files,
                i,
                quality_threshold,
                result_files,
            ): i
            for i in range(n_runs)
        }
        for future in concurrent.futures.as_completed(future_to_url):
            i = future_to_url[future]
            try:
                name = future.result()
            except Exception as exception:
                print(f"{event_log_files} run_{i} - {exception}")
            else:
                print(f"{event_log_files} run_{i} - {name}")

### Invoke Lambda function

In [ ]:
!aws sso login --profile=dev-poweruser

In [ ]:
import boto3

boto3.setup_default_session(profile_name="dev-poweruser")
lambda_client = boto3.client("lambda")

n_runs = 10

# Define the quality threshold
quality_threshold = 0.9


for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_name = (
        "backward/scenarios/doe/"
        + f"merge_after_steps={'-'.join(merge_after_steps)}"
        + f"+split_after_steps={'-'.join(split_after_steps)}"
        + f"+number_of_devices={'-'.join(number_of_devices_label)}"
        + f"+n_resources_factor={n_resources_factor}"
    )

    split_configs = [
        {"after_step": step, "number_of_split_lots": 2} for step in split_after_steps
    ]

    event = {
        "quality_threshold": 0.9,
        "scenario_name": scenario_name,
        "scenario": generate_scenario(
            output_file=scenario_name + ".json",
            n_lots=32,
            n_devices=number_of_devices,
            steps_resources={
                "WT": (n_resources_factor * 2, 1),
                "DB": (n_resources_factor * 2, 2),
                "WB": (n_resources_factor * 4, 20),
                "Marking": (n_resources_factor * 4, 1),
                "FT": (n_resources_factor * 2, 1),
            },
            root_cause_resource="WB1",
            # required_material: Dict[str, str] = None,
            merge_after_steps=list(merge_after_steps),
            # n_lots_merge: int = 2,
            split_configs=split_configs,
        ),
    }

    count = 0
    for i in range(n_runs):
        event["run_number"] = i
        lambda_response = lambda_client.invoke(
            FunctionName="research-trace-backward-experiments",
            InvocationType="Event",  # RequestResponse
            # LogType="Tail",
            Payload=json.dumps(event).encode(),
        )

        if lambda_response["ResponseMetadata"]["HTTPStatusCode"] == 202:
            count += 1

    print(f"{scenario_name}: {count} invocations")

### Retrieve results from S3

In [ ]:
from botocore.exceptions import ClientError

s3_client = boto3.client("s3")

for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_name = (
        "backward/scenarios/doe/"
        + f"merge_after_steps={'-'.join(merge_after_steps)}"
        + f"+split_after_steps={'-'.join(split_after_steps)}"
        + f"+number_of_devices={'-'.join(number_of_devices_label)}"
        + f"+n_resources_factor={n_resources_factor}"
    )

    result_files = Path(f"output/{scenario_name}/")
    if not result_files.exists():
        os.mkdir(result_files)

    for i in range(n_runs):
        object_key = os.path.join(
            "research", "trace-backward-experiments", scenario_name, f"run_{i}.json.gz"
        ).replace("\\", "/")
        file_name = os.path.join("output", scenario_name, f"run_{i}.json.gz")

        if Path(file_name).exists():
            print(f"{file_name} already exists")
            continue

        try:
            s3_client.download_file(
                Bucket="574885398813-spill-data", Key=object_key, Filename=file_name
            )
        except ClientError as exception:
            print(f"Failed retrieving: {object_key}: {exception}")

# Process results

## Compute steps to find root cause (entity)

In [ ]:
def compute_n_steps(probability_dicts, root_cause_entity) -> DataFrame:
    group_key = "product_model"

    results = []
    for event_data_file, probability_dict in probability_dicts.items():
        df_trace = DataFrame(probability_dict)

        for group_value in df_trace[group_key].unique():
            # Aggregate to get probabilities for target entities
            df_trace_grouped = (
                df_trace[df_trace[group_key] == group_value]
                .groupby(["entity_source", "entity_target"])
                .agg(
                    {
                        "probability": "sum",
                        "n_merges": "sum",
                        "split_merge_kl": "mean",
                        "n_production_steps_aggregations": list,
                    }
                )
            )
            df_trace_grouped.reset_index(inplace=True)

            # Remove lists with NaN value
            df_trace_grouped["n_production_steps_aggregations"] = df_trace_grouped[
                "n_production_steps_aggregations"
            ].apply(lambda x: [] if any(not isinstance(i, list) for i in x) else x)

            entities_source = df_trace_grouped["entity_source"].unique()
            for entity_source in entities_source:
                df_selected = df_trace_grouped[
                    df_trace_grouped["entity_source"] == entity_source
                ]

                # Shuffle/sample DataFrame for random order of inspection
                n_steps_random = get_entity_record_number(
                    df_selected.sample(frac=1), root_cause_entity
                )

                # Sort values for informed order of inspection
                n_steps_sorted = get_entity_record_number(
                    df_selected.sort_values("probability", ascending=False),
                    root_cause_entity,
                )

                results.append(
                    {
                        "file": event_data_file,
                        group_key: group_value,
                        "probabilities": df_selected.to_dict(orient="records"),
                        "n_steps_random": n_steps_random,
                        "n_steps_sorted": n_steps_sorted,
                    }
                )

    return DataFrame(results)

In [ ]:
import gzip

from pandas import read_csv

# processed_scenarios = read_csv("output/backward/combined_report.csv")[
#     "scenario_name"
# ].unique()
processed_scenarios = []

scenario_names = []
for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_names.append(
        (
            "backward/scenarios/doe/"
            + f"merge_after_steps={'-'.join(merge_after_steps)}"
            + f"+split_after_steps={'-'.join(split_after_steps)}"
            + f"+number_of_devices={'-'.join(number_of_devices_label)}"
            + f"+n_resources_factor={n_resources_factor}"
        )
    )

scenario_dict = {}
for scenario_name in scenario_names:
    # Skip scenarios that are already processed
    if scenario_name in processed_scenarios:
        continue

    try:
        with open(scenario_name + ".json") as f:
            config = json.load(f)
    except FileNotFoundError:
        print("Skip: ", scenario_name)
        continue

    print(scenario_name)

    lower_yield_resources = [
        resource["id"]
        for resource in config["production_resources"]
        if resource["process_yield"] < 1
    ]
    if len(lower_yield_resources) > 1:
        print("Multiple resources with yield < 1")

    root_cause_entity = (
        f"http://example.org/id/ekg/aggregated_traces/{lower_yield_resources[0]}"
    )

    probability_dicts = {}
    for result_file in Path(f"output/{scenario_name}/").glob("*.json.gz"):
        with gzip.open(result_file) as f:
            probability_dicts[result_file] = json.load(f)

    scenario_dict[scenario_name] = compute_n_steps(probability_dicts, root_cause_entity)

print(len(scenario_dict))

In [ ]:
from pandas import concat, isna

df_combined = DataFrame()
# df_combined = read_csv("output/backward/combined_report.csv")
# df_combined.drop(index=df_combined[df_combined["scenario_name"].str.contains("=Marking")].index, inplace=True)
# df_combined["probabilities"] = df_combined["probabilities"].apply(json.loads)

for k, v in scenario_dict.items():
    v["scenario_name"] = k
    df_combined = concat([df_combined, v])

df_combined["n_steps_diff"] = (
    df_combined["n_steps_random"] - df_combined["n_steps_sorted"]
)

df_combined["probabilities"] = df_combined["probabilities"].apply(json.dumps)
df_combined.to_csv("output/backward/combined_report_DoE.csv", index=False)

## Visualize results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import isna, read_csv, set_option
from scipy.stats import pearsonr

set_option("display.max_colwidth", None)

In [ ]:
df_combined = read_csv("output/backward/combined_report_DoE.csv")

In [ ]:
# If "n_steps_random" is NaN, the root cause entity does not occur in the trace
df_combined[~isna(df_combined["n_steps_random"])].describe()

In [ ]:
df_combined.groupby("scenario_name")[
    ["n_steps_diff", "n_steps_random", "n_steps_sorted"]
].describe().T

In [ ]:
from pandas import notna
from pandas.api.types import CategoricalDtype


def combine_steps(row):
    steps_order = ["WT", "DB", "WB", "Marking", "FT"]
    merge_steps = (
        row["merge_after_steps"].split("-") if notna(row["merge_after_steps"]) else []
    )
    split_steps = (
        row["split_after_steps"].split("-") if notna(row["split_after_steps"]) else []
    )

    combined = []
    for step in steps_order:
        if step in merge_steps:
            combined.append(f"{step}-merge")
        elif step in split_steps:
            combined.append(f"{step}-split")
        else:
            combined.append(f"{step}")

    return "-".join(combined)


def merge_split_order(row):
    i_merge = row.find("merge")
    i_split = row.find("split")

    return i_merge < i_split


for parameter in [
    "merge_after_steps",
    "split_after_steps",
    "number_of_devices",
    "n_resources_factor",
]:
    df_combined[parameter] = df_combined["scenario_name"].str.extract(
        rf"{parameter}=([A-Za-z0-9-]+)"
    )


df_combined["merge_split_steps"] = df_combined.apply(combine_steps, axis=1)
df_combined["merge_before_split"] = df_combined["merge_split_steps"].apply(
    merge_split_order
)
df_combined["count_merge"] = df_combined["merge_split_steps"].str.count("merge")
df_combined["count_split"] = df_combined["merge_split_steps"].str.count("split")


cat_types = {
    "number_of_devices": CategoricalDtype(
        categories=["50", "75-25", "90-10", "125-25", "170-10"], ordered=True
    ),
}
for c in df_combined.columns:
    if c in cat_types:
        df_combined[c] = df_combined[c].astype(cat_types[c])

In [ ]:
# Create a pivot table for interaction profile
interaction_profile = df_combined.pivot_table(
    values="n_steps_diff",
    index="merge_split_steps",
    columns=["number_of_devices", "n_resources_factor"],
    aggfunc="median",
).loc[:, (["50", "75-25", "90-10", "125-25", "170-10"], ["2", "4", "8", "16"])]

# Plot the heatmap
plt.figure(figsize=(20, 40))
sns.heatmap(
    interaction_profile,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar_kws={"label": "Mean n_steps_diff"},
)
plt.title("Interaction Profile for n_steps_diff")
plt.xlabel("Number of Devices and Resources Factor")
plt.ylabel("Merge and Split Steps")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
from pandas.plotting import parallel_coordinates

import matplotlib.pyplot as plt

columns = [
    "count_merge",
    "count_split",
    "merge_before_split",
    "number_of_devices",
    "n_resources_factor",
]

# Filter the data to ensure no NaN values in the required columns
df_plot = df_combined.dropna(subset=columns + ["n_steps_diff"]).copy()
# df_plot["number_of_devices_map"]

for c in columns:
    # df_plot[c] = df_plot[c].astype(str)

    if CategoricalDtype.is_dtype(df_combined[c]):
        df_plot[c] = df_plot[c].cat.codes

    df_plot[c] = df_plot[c].astype(float)
    # min_val = df_plot[c].min()
    # max_val = df_plot[c].max()
    # df_plot[c] = (df_plot[c] - min_val) / (max_val - min_val)

# Create the parallel coordinates plot
plt.figure(figsize=(15, 8))
parallel_coordinates(
    df_plot[columns + ["n_steps_diff"]],
    class_column="n_steps_diff",
    colormap=plt.cm.coolwarm,
)

for c in columns:
    if CategoricalDtype.is_dtype(df_combined[c]):
        values_labels = enumerate(df_combined[c].dtype.categories)
    else:
        values_labels = [(float(v), v) for v in sorted(df_combined[c].unique())]

    for value, label in values_labels:
        plt.annotate(
            label, xy=(columns.index(c), value), ha="left", va="center"
        )

# Add labels and title
plt.title("Parallel Coordinates Plot for n_steps_diff")
plt.xlabel("Factors")
plt.ylabel("Factor value (normalized)")
plt.xticks(rotation=45)
plt.grid(True)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
parallel_coordinates(
    df_plot[columns + ["n_steps_diff"]],
    class_column="n_steps_diff",
    colormap=plt.cm.coolwarm,
)

c = "number_of_devices"
for i, label in enumerate(cat_type.categories):
    plt.annotate(
        label, xy=(columns.index("number_of_devices"), i), ha="left", va="center"
    )

In [ ]:
plt.figure(figsize=(20, 15))
sns.set_theme(font_scale=1)

df_plot = df_combined[
    df_combined["scenario_name"].str.contains("n_resources_factor=1")
].copy()
# df_plot = df_combined.copy()

x_label = "split_after_steps"
df_plot[x_label] = df_plot["scenario_name"].str.extract(rf"{x_label}=([A-Za-z0-9-]+)")
df_plot[x_label] = df_plot[x_label].fillna("-")

group_label = "number_of_devices"
# group_label = "n_resources_factor"
# group_label = "merge_after_steps"
df_plot[group_label] = df_plot["scenario_name"].str.extract(
    rf"{group_label}=([A-Za-z0-9-]+)"
)

ax = sns.violinplot(
    data=df_plot.dropna(subset=["n_steps_diff"]),
    x=x_label,
    y="n_steps_diff",
    hue=group_label,
)

ax.set_ylabel("$R$")
ax.tick_params(axis="x", rotation=90)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

# plt.grid()

# plt.savefig(
#     f"output/backward/figures/violin_{x_label}_by_{group_label}.svg", bbox_inches="tight"
# )

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

scenario_options = [
    "merge_after_steps",
    "number_of_devices",
    "n_resources_factor",
    "split_after_steps",
]

y_label_mapping = {
    "n_steps_diff": "$R$",
}

# Define interactive widgets
filter_scenario_widget = widgets.Text(
    value="",
    description="Filter Scenario:",
    layout=widgets.Layout(width="50%"),
)

x_label_widget = widgets.Dropdown(
    options=scenario_options,
    value="merge_after_steps",
    description="X-axis Label:",
)

y_labels_widget = widgets.SelectMultiple(
    options=df_combined.columns,
    value=["n_steps_diff"],
    description="Y-axis Label:",
)

group_label_widget = widgets.Dropdown(
    options=scenario_options,
    value="merge_after_steps",
    description="Group Label:",
)


# Function to update the plot
def update_plot(filter_scenario, x_label, y_labels, group_label):
    clear_output(wait=True)
    display(
        filter_scenario_widget,
        x_label_widget,
        y_labels_widget,
        group_label_widget,
        plot_button,
    )

    df_plot = df_combined[
        df_combined["scenario_name"].str.contains(filter_scenario)
    ].copy()

    df_plot[x_label] = (
        df_plot["scenario_name"].str.extract(rf"{x_label}=([A-Za-z0-9-]+)").fillna("")
    )
    df_plot[group_label] = (
        df_plot["scenario_name"]
        .str.extract(rf"{group_label}=([A-Za-z0-9-]+)")
        .fillna("")
    )

    fig, axes = plt.subplots(nrows=len(y_labels), sharex=True, figsize=(20, 15))
    sns.set_theme(font_scale=1)

    for i, y_label in enumerate(y_labels):
        ax = sns.boxplot(
            data=df_plot[~isna(df_plot["n_steps_diff"])],
            x=x_label,
            y=y_label,
            hue=group_label,
            ax=axes[i] if isinstance(axes, list) else axes,
            legend="full",
        )
        ax.tick_params(axis="x", rotation=90)
        ax.set_ylabel(y_label_mapping.get(y_label, y_label))

    # plt.savefig(
    #     f"output/backward/figures/violin_{'+'.join(y_labels)}_x={x_label}_group={group_label}.svg",
    #     bbox_inches="tight",
    # )
    plt.show()


# Button to trigger the update
plot_button = widgets.Button(description="Update Plot")


# Event handler for the button
def on_button_click(b):
    update_plot(
        filter_scenario_widget.value,
        x_label_widget.value,
        y_labels_widget.value,
        group_label_widget.value,
    )


plot_button.on_click(on_button_click)

# Display widgets
display(
    filter_scenario_widget,
    x_label_widget,
    y_labels_widget,
    group_label_widget,
    plot_button,
)

## Visualize trace

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

visualize_ekg_graph_rdf = load_rdf_graph(
    basepath.joinpath(
        "logs/backward/scenarios/doe/merge_after_steps=FT+split_after_steps=WT-DB-WB-Marking+number_of_devices=10-90+n_resources_factor=8/run_0.json"
    ),
    store="Oxigraph",
)

# Insert DF/DP relations
visualize_ekg_graph_rdf = insert_DF_DP(visualize_ekg_graph_rdf)

# Insert fractions on the relations
visualize_ekg_graph_rdf = insert_fractions(visualize_ekg_graph_rdf)

# Create NetworkX graph
visualize_ekg_graph_nx = generate_networkx_di_graph(visualize_ekg_graph_rdf)

generate_graph_visualization(visualize_ekg_graph_nx, base_figure_path="output/check")

In [ ]:
scenario_name = "backward/scenarios/doe/merge_after_steps=FT+split_after_steps=WT-DB-WB-Marking+number_of_devices=10-90+n_resources_factor=8"
try:
    with open(scenario_name + ".json") as f:
        config = json.load(f)
except FileNotFoundError:
    print("Skip: ", scenario_name)

lower_yield_resources = [
    resource["id"]
    for resource in config["production_resources"]
    if resource["process_yield"] < 1
]
if len(lower_yield_resources) > 1:
    print("Multiple resources with yield < 1")

root_cause_entity = (
    f"http://example.org/id/ekg/aggregated_traces/{lower_yield_resources[0]}"
)

probability_dicts = {}
for result_file in Path(f"output/{scenario_name}/").glob("*.json"):
    with open(result_file) as f:
        probability_dicts[result_file] = json.load(f)

df = compute_n_steps(probability_dicts, root_cause_entity)

In [ ]:
(df["n_steps_random"] - df["n_steps_sorted"]).mean()

In [ ]:
from pandas import read_json

read_json(
    "output/backward/scenarios/doe/merge_after_steps=FT+split_after_steps=WT-DB-WB-Marking+number_of_devices=10-90+n_resources_factor=8/run_0.json"
)["split_merge_kl"].mean()

In [ ]:
visualize_ekg_graph_rdf.query("""
PREFIX : <http://example.org/def/ekg/aggregated_traces/>

SELECT DISTINCT
  ?event
  ?type
  ?product
  ?sum_amount_in
  (replace(?entity_amount_in, "http://example.org/id/ekg/aggregated_traces/", "") as ?_entity_amount_in)
  ?sum_amount_out
  (replace(?entity_amount_out, "http://example.org/id/ekg/aggregated_traces/", "") as ?_entity_amount_out)
  (?sum_amount_in=?sum_amount_out as ?equal)
WHERE {
  { SELECT ?event ?product ?type (sum(xsd:float(strafter(?entity_amount_in, "|"))) as ?sum_amount_in) (group_concat(?entity_amount_in ; separator=", ") as ?entity_amount_in) {
    # Pipe separator (https://www.rfc-editor.org/rfc/rfc3986)
    { SELECT ?event ?product ?type (concat(str(?entity), "|", str(?amount_in)) as ?entity_amount_in) {
      VALUES ?type {
        :DirectlyFollows_AggregatedEntity
        :DirectlyPrecedes_AggregatedEntity
      }
      ?relation a ?type ;
        :target ?event ;
        :amount ?amount_in ;
        :class ?entity .

      ?entity a :AggregatedEntity .

      OPTIONAL {
        ?relation :class ?product .
        ?product a :Product .
      }
    }}
  } GROUP BY ?event ?product ?type }

  { SELECT ?event ?product ?type (sum(xsd:float(strafter(?entity_amount_out, "|"))) as ?sum_amount_out) (group_concat(?entity_amount_out ; separator=", ") as ?entity_amount_out) {
    { SELECT ?event ?product ?type (concat(str(?entity), "|", str(?amount_out)) as ?entity_amount_out) {
      VALUES ?type {
        :DirectlyFollows_AggregatedEntity
        :DirectlyPrecedes_AggregatedEntity
      }
      ?relation a ?type ;
        :source ?event ;
        :amount ?amount_out ;
        :class ?entity .

      ?entity a :AggregatedEntity .

      OPTIONAL {
        ?relation :class ?product .
        ?product a :Product .
      }
    }}
  } GROUP BY ?event ?product ?type }
}
ORDER BY ?type DESC(?equal)""").bindings